Import required library like Langchain, FAIss, Gradio.

In [ ]:
# Installing required packages using the kernel's own Python
import sys
!{sys.executable} -m pip install --force-reinstall packaging
!{sys.executable} -m pip install --force-reinstall gradio gradio_client
!{sys.executable} -m pip install langchainhub langchain-openai langchain beautifulsoup4
!{sys.executable} -m pip install langchain-community faiss-cpu tavily-python


In [20]:
# Necessary Imports
# import kagglehub
import csv
import pandas as pd
import math
import numpy as np
import os
import getpass
from langchain_core.output_parsers import StrOutputParser
# from google.colab import drive

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
# from langchain.chains.combine_documents import create_stuff_documents_chain
# from langchain.chains import create_retrieval_chain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser

In [21]:
# dataset = pd.read_csv('DataSet/dataset.csv')
# symptom_description = pd.read_csv('DataSet/symptom_description.csv')
# symptom_precaution = pd.read_csv('DataSet/symptom_precaution.csv')
# symptom_severity = pd.read_csv('DataSet/symptom-severity.csv')

dataset = pd.read_csv('DataSet/dataset.csv')
symptom_description = pd.read_csv('DataSet/symptom_Description.csv')   # capital D
symptom_precaution = pd.read_csv('DataSet/symptom_precaution.csv')
symptom_severity = pd.read_csv('DataSet/Symptom-severity.csv')          # capital S

In [ ]:
# dataset.head()
# #disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Severity
# dataset.info()
# symptom_severity.head()
# symptom_severity.info()
# symptom_description.info()
# symptom_description.head()
# symptom_precaution.info()
# symptom_precaution.head(10)

In [22]:
dataset = dataset.rename(columns={'disease': 'Disease'})
symptom_description = symptom_description.rename(columns={'disease':'Disease'})
symptom_precaution = symptom_precaution.rename(columns={'disease':'Disease'})
symptom_severity = symptom_severity.rename(columns={'disease':'Disease'})

In [23]:
# Melt the symptom columns into a single column for easier merging
symptom_cols = [col for col in dataset.columns if col.startswith('Symptom')]
dataset_melted = dataset.melt(id_vars=['Disease'], value_vars=symptom_cols, value_name='Symptom').drop(columns='variable')
dataset_melted = dataset_melted.dropna(subset=['Symptom'])
dataset_melted['Symptom'] = dataset_melted['Symptom'].str.strip()

In [ ]:
# dataset_melted.head()

In [24]:
# Merge with symptom severity
symptom_severity['Symptom'] = symptom_severity['Symptom'].str.strip()
dataset_with_severity = dataset_melted.merge(symptom_severity, on='Symptom', how='left')

In [ ]:
# dataset_with_severity.head()

In [25]:
# Merge with symptom description (disease-level)
dataset_with_desc = dataset_with_severity.merge(symptom_description, on='Disease', how='left')

In [ ]:
# dataset_with_desc.head()

In [26]:
# Merge with symptom precaution (disease-level)
dataset_with_precaution = dataset_with_desc.merge(symptom_precaution, on='Disease', how='left')

In [ ]:
# dataset_with_precaution.head()

In [27]:
# Group by disease to create a RAG-friendly text document per disease
def create_rag_document(group):
    disease = group.name  # group.name is the Disease value (groupby key)
    symptoms = group['Symptom'].tolist()
    severities = group['weight'].tolist()
    description = group['Description'].iloc[0] if 'Description' in group.columns else 'N/A'
    
    precaution_cols = [col for col in group.columns if col.lower().startswith('precaution')]
    precautions = group[precaution_cols].iloc[0].dropna().tolist()
    
    symptom_severity_info = ', '.join([f"{s} (severity: {w})" for s, w in zip(symptoms, severities)])
    precaution_info = ' | '.join(precautions)
    
    document = (
        f"Disease: {disease}\n"
        f"Description: {description}\n"
        f"Symptoms with Severity: {symptom_severity_info}\n"
        f"Precautions: {precaution_info}"
    )
    return pd.Series({'Disease': disease, 'Document': document})


In [28]:
rag_dataset = dataset_with_precaution.groupby('Disease').apply(create_rag_document, include_groups=False).reset_index(drop=True)


In [29]:
print(f"Total RAG documents: {len(rag_dataset)}")
print("\nSample Document:\n")
print(rag_dataset['Document'].iloc[0])


Total RAG documents: 41

Sample Document:

Disease: (vertigo) Paroymsal  Positional Vertigo
Description: Benign paroxysmal positional vertigo (BPPV) is one of the most common causes of vertigo — the sudden sensation that you're spinning or that the inside of your head is spinning. Benign paroxysmal positional vertigo causes brief episodes of mild to intense dizziness.
Symptoms with Severity: vomiting (severity: 5.0), vomiting (severity: 5.0), headache (severity: 3.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), headache (severity: 3.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), vomiting (severity: 5.0), headache (severity: 3.0), vomitin

In [30]:
# Create text chunks from RAG documents using LangChain
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    length_function=len,
)

# Extract documents as list of strings
documents = rag_dataset['Document'].tolist()

# Split documents into chunks
chunks = text_splitter.create_documents(
    texts=documents,
    metadatas=[{'Disease': disease} for disease in rag_dataset['Disease'].tolist()]
)

print(f"Total chunks created: {len(chunks)}")
print("\nSample Chunk:\n")
print(chunks[0].page_content)
print("\nChunk Metadata:", chunks[0].metadata)

Total chunks created: 5710

Sample Chunk:

Disease: (vertigo) Paroymsal  Positional Vertigo

Chunk Metadata: {'Disease': '(vertigo) Paroymsal  Positional Vertigo'}


In [33]:
# Set OpenAI API key
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPEN_AI_KEY")
if not api_key:
	api_key = getpass.getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# getpass.getpass("Enter your OpenAI API key: ")

# Create embeddings
embeddings = OpenAIEmbeddings()

# Create FAISS vector store from chunks
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save the vector store locally
vectorstore.save_local("faiss_disease_index")

print("FAISS vector store created and saved successfully!")
print(f"Total vectors stored: {vectorstore.index.ntotal}")

FAISS vector store created and saved successfully!
Total vectors stored: 5710


In [34]:
# Create a generic agent to answer questions.
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Create retriever from existing vectorstore
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

# Initialize LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Create prompt template for diagnosis
diagnosis_prompt = ChatPromptTemplate.from_template("""
You are a medical assistant AI that helps diagnose diseases based on symptoms.
Use ONLY the provided context to answer. Do not make up information.

Context from medical database:
{context}

User's symptoms: {input}

Please provide:
1. **Most Likely Disease(s)**: Based on the symptoms described
2. **Description**: Brief explanation of the disease
3. **Matching Symptoms**: Which reported symptoms match
4. **Severity Assessment**: Based on symptom severity scores
5. **Recommended Precautions**: Steps the user should take

Important: Always recommend consulting a healthcare professional for proper diagnosis.
""")

# Create document chain
document_chain = create_stuff_documents_chain(llm, diagnosis_prompt)

# Create retrieval chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [35]:
# Diagnosis function
def diagnose_symptoms(symptoms_input):
    """
    Takes user symptoms as input and returns diagnosis using RAG.
    
    Args:
        symptoms_input (str): User-described symptoms
    
    Returns:
        str: Diagnosis response
    """
    response = retrieval_chain.invoke({"input": symptoms_input})
    return response["answer"]

# Test the diagnosis tool
test_symptoms = "I have itching, skin rash, and nodal skin eruptions"
print("=== Symptom Diagnosis Tool ===\n")
print(f"Symptoms: {test_symptoms}\n")
print("Diagnosis:\n")
result = diagnose_symptoms(test_symptoms)
print(result)

=== Symptom Diagnosis Tool ===

Symptoms: I have itching, skin rash, and nodal skin eruptions

Diagnosis:

1. **Most Likely Disease(s)**: Dermatitis
2. **Description**: Dermatitis is a common skin condition characterized by inflammation of the skin, leading to symptoms such as itching, skin rash, and skin eruptions.
3. **Matching Symptoms**: Itching, skin rash, nodal skin eruptions
4. **Severity Assessment**: The severity scores for itching, skin rash, and nodal skin eruptions range from 1.0 to 4.0, indicating a moderate level of symptoms.
5. **Recommended Precautions**: The user should avoid scratching the affected areas, keep the skin moisturized, and consult a dermatologist for proper diagnosis and treatment.


# MediBot aims to assist users by:
* Providing probable disease predictions based on symptom inputs.
* Offering symptom severity analysis to guide users on urgency.
* Explaining diseases in a simplified manner.
* Suggesting preventive measures and self-care recommendations.

In [95]:
# Creating a multi-tool medical assistant agent with Gradio interface
from langchain.tools import tool

# Tool 1: Disease Prediction based on symptoms
@tool
def predict_disease(symptoms: str) -> str:
    """Predicts probable diseases based on user-described symptoms using RAG."""
    prediction_prompt = ChatPromptTemplate.from_template("""
    You are a medical assistant AI. Based on the symptoms provided by the user, predict the most probable disease(s).
    Use ONLY the provided context from the medical database.

    Context:
    {context}

    Symptoms: {input}

    Provide:
    1. **Most Probable Disease(s)**: List top 1-3 diseases
    2. **Confidence**: High/Medium/Low based on symptom match
    3. **Key Matching Symptoms**: Which symptoms led to this prediction

    Always recommend consulting a healthcare professional.
    """)
    chain = prediction_prompt | llm | StrOutputParser()
    docs = retriever.invoke(symptoms)
    context = "\n\n".join([doc.page_content for doc in docs])
    return chain.invoke({"context": context, "input": symptoms})


# Tool 2: Symptom Severity Analysis
@tool
def analyze_symptom_severity(symptoms: str) -> str:
    """Analyzes the severity of given symptoms to guide urgency of medical attention."""
    symptom_list = [s.strip().lower().replace(" ", "_") for s in symptoms.replace(",", " ").split()]
    severity_data = symptom_severity[symptom_severity['Symptom'].str.lower().isin(symptom_list)]
    
    if severity_data.empty:
        severity_info = "No exact severity data found for provided symptoms."
    else:
        severity_info = severity_data.to_string(index=False)
    
    severity_prompt = ChatPromptTemplate.from_template("""
    You are a medical assistant AI. Analyze the severity of the symptoms based on the severity scores below.
    Severity scale: 1-3 = Mild, 4-5 = Moderate, 6-7 = Severe

    Severity Data from Database:
    {severity_data}

    Context from medical database:
    {context}

    User Symptoms: {input}

    Provide:
    1. **Severity Level per Symptom**: List each symptom with its severity score and category (Mild/Moderate/Severe)
    2. **Overall Urgency**: Low / Medium / High urgency for seeking medical care
    3. **Urgency Reasoning**: Brief explanation of the urgency assessment
    4. **Immediate Action**: What the user should do right now

    Always recommend consulting a healthcare professional for proper evaluation.
    """)
    chain = severity_prompt | llm | StrOutputParser()
    docs = retriever.invoke(symptoms)
    context = "\n\n".join([doc.page_content for doc in docs])
    return chain.invoke({"context": context, "severity_data": severity_info, "input": symptoms})


# Tool 3: Disease Explanation in Simple Terms
@tool
def explain_disease(disease_name: str) -> str:
    """Explains a disease in simplified, easy-to-understand terms for a general audience."""
    explanation_prompt = ChatPromptTemplate.from_template("""
    You are a friendly medical educator. Explain the disease in simple, easy-to-understand language
    for someone with no medical background. Use ONLY the context provided.

    Context from medical database:
    {context}

    Disease to explain: {input}

    Provide:
    1. **What is it?**: Simple one-paragraph explanation
    2. **Who gets it?**: Common risk factors in plain language
    3. **Common Symptoms**: List symptoms in everyday language
    4. **Is it serious?**: General outlook
    5. **Fun Fact / Key Takeaway**: One memorable fact about the disease

    Use simple words, avoid medical jargon.
    """)
    chain = explanation_prompt | llm | StrOutputParser()
    docs = retriever.invoke(disease_name)
    context = "\n\n".join([doc.page_content for doc in docs])
    return chain.invoke({"context": context, "input": disease_name})


# Tool 4: Preventive Measures and Self-Care Recommendations
@tool
def suggest_precautions(symptoms_or_disease: str) -> str:
    """Suggests preventive measures and self-care recommendations based on symptoms or disease name."""
    precaution_prompt = ChatPromptTemplate.from_template("""
    You are a medical assistant AI focused on preventive care and self-care guidance.
    Use ONLY the provided context from the medical database.

    Context from medical database:
    {context}

    User's symptoms or disease: {input}

    Provide:
    1. **Recommended Precautions**: List specific precautions from the database
    2. **Self-Care Steps**: Practical daily self-care steps
    3. **Lifestyle Modifications**: Diet, exercise, sleep recommendations if relevant
    4. **Warning Signs**: Symptoms that indicate immediate medical attention is needed
    5. **Follow-Up**: When and how to follow up with a healthcare provider

    Always remind the user to consult a healthcare professional before making medical decisions.
    """)
    chain = precaution_prompt | llm | StrOutputParser()
    docs = retriever.invoke(symptoms_or_disease)
    context = "\n\n".join([doc.page_content for doc in docs])
    return chain.invoke({"context": context, "input": symptoms_or_disease})




In [96]:
# Test all tools
print("=" * 60)
print("Testing MediBot Tools")
print("=" * 60)

test_input = "I have fever, headache, and joint pain"

print("\n[Tool 1] Disease Prediction:")
print(predict_disease.invoke(test_input))

print("\n[Tool 2] Symptom Severity Analysis:")
print(analyze_symptom_severity.invoke(test_input))

print("\n[Tool 3] Disease Explanation:")
print(explain_disease.invoke("Dengue"))

print("\n[Tool 4] Precautions & Self-Care:")
print(suggest_precautions.invoke(test_input))

Testing MediBot Tools

[Tool 1] Disease Prediction:
1. **Most Probable Disease(s)**: Dengue fever
2. **Confidence**: High
3. **Key Matching Symptoms**: High fever, headache

[Tool 2] Symptom Severity Analysis:
1. **Severity Level per Symptom**:
   - Fever: 7.0 (Severe)
   - Headache: 3.0 (Mild)
   - Joint pain: Not provided in the database

2. **Overall Urgency**: High urgency for seeking medical care

3. **Urgency Reasoning**: The presence of a severe symptom like high fever indicates a potentially serious underlying condition that requires immediate medical attention. Additionally, the combination of fever, headache, and joint pain could be indicative of various infections or inflammatory conditions that need prompt evaluation and treatment.

4. **Immediate Action**: It is recommended to seek medical attention immediately. Contact a healthcare provider or visit an urgent care facility for a thorough evaluation and appropriate management of your symptoms.

[Tool 3] Disease Explanation

In [97]:
#Creating REACT Agent that can reason step-by-step and pick the right tools based on user queries

from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic import hub
from langchain_core.prompts import PromptTemplate
from langchain_classic.memory import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Define the tools list
tools = [predict_disease, analyze_symptom_severity, explain_disease, suggest_precautions]

# ReAct prompt template - improved for better parsing
react_prompt_template = PromptTemplate.from_template("""You are MediBot, an intelligent medical assistant AI that uses reasoning and tools to help users.

You have access to the following tools:
{tools}

Use the following format exactly:

Question: the input question you must answer
Thought: you should always think about what to do and which tool to use
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: Write your answer as natural, conversational English without JSON or structured formatting.

Guidelines:
- Use 'predict_disease' for symptom descriptions and disease prediction
- Use 'analyze_symptom_severity' to assess symptom urgency
- Use 'explain_disease' to explain specific diseases
- Use 'suggest_precautions' for prevention advice
- You may chain multiple tools for comprehensive answers
- Always remind users to consult healthcare professionals

Begin!

Previous conversation history:{chat_history}

Question: {input}
Thought:{agent_scratchpad}""")

# Initialize a chat message history object with a session ID
# This stores the conversation history for a given session, allowing stateful interactions.
memory = ChatMessageHistory(session_id="test-session")

# Initialize ReAct agent LLM (use gpt-4 for better reasoning if available)
react_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Create the ReAct agent
react_agent = create_react_agent(
    llm=react_llm,
    tools=tools,
    prompt=react_prompt_template
)

# Create the agent executor to run the ReAct agent with tool access and memory
react_agent_executor = AgentExecutor(
    agent=react_agent,
    tools=tools,
    verbose=True,
    max_iterations=10,  # Increased from 6 to allow more attempts
    handle_parsing_errors=True,
    return_intermediate_steps=True
)

# Initialize an Agent with Chat History to maintain conversation context
react_agent_with_chat_history = RunnableWithMessageHistory(
    react_agent_executor,  # The main AgentExecutor responsible for executing queries and invoking tools.

    lambda session_id: memory,

    input_messages_key="input",  # Specifies the key in the input dictionary where the user query is stored.

    history_messages_key="chat_history",  # Specifies the key under which the conversation history is stored.
)

In [94]:
react_agent_with_chat_history.invoke(
    {"input": "I have high fever, joint pain, and rashes. What disease could this be and how serious is it?"},
    config={"configurable": {"session_id": "test-session"}},
)



> Entering new AgentExecutor chain...
The user is describing symptoms and wants to know the probable disease and its severity. I should first predict the disease based on the symptoms and then analyze the severity of the symptoms.
Action: predict_disease
Action Input: "high fever, joint pain, rashes"1. **Most Probable Disease(s)**: Dengue Fever
2. **Confidence**: High
3. **Key Matching Symptoms**: High fever, joint pain, skin rashThe predicted disease based on the symptoms you provided is Dengue Fever. Now, I should analyze the severity of these symptoms to guide the urgency of seeking medical attention.
Action: analyze_symptom_severity
Action Input: "high fever, joint pain, rashes"1. **Severity Level per Symptom**:
   - High fever (severity: 7.0) - Severe
   - Joint pain (severity: 3.0) - Mild
   - Rashes (severity: 3.0) - Mild

2. **Overall Urgency**: High urgency for seeking medical care

3. **Urgency Reasoning**: The presence of a severe symptom like high fever indicates a potent

{'input': 'I have high fever, joint pain, and rashes. What disease could this be and how serious is it?',
 'chat_history': [],
 'output': 'Based on the symptoms of high fever, joint pain, and rashes you described, the most probable disease is Dengue Fever. The severity of these symptoms indicates a high urgency for seeking medical care. It is crucial that you consult a healthcare professional immediately for a thorough evaluation and appropriate treatment.',
 'intermediate_steps': [(AgentAction(tool='predict_disease', tool_input='high fever, joint pain, rashes', log='The user is describing symptoms and wants to know the probable disease and its severity. I should first predict the disease based on the symptoms and then analyze the severity of the symptoms.\nAction: predict_disease\nAction Input: "high fever, joint pain, rashes"'),
   '1. **Most Probable Disease(s)**: Dengue Fever\n2. **Confidence**: High\n3. **Key Matching Symptoms**: High fever, joint pain, skin rash'),
  (AgentAction

In [98]:
# Initialize session-based chat history
session_memory = {}

def get_memory(session_id):
    """Fetch or create a chat history instance for a given session."""
    if session_id not in session_memory:
        session_memory[session_id] = ChatMessageHistory()
    return session_memory[session_id]


In [99]:
# Helper function to convert JSON to natural English
import re


def clean_response(text):
    """
    Convert JSON format or structured responses to natural English text.
    """
    try:
        # Try to parse as JSON
        if text.strip().startswith('{') or text.strip().startswith('['):
            data = json.loads(text)
            if isinstance(data, dict):
                # Convert dict to readable paragraphs
                paragraphs = []
                for key, value in data.items():
                    if isinstance(value, list):
                        formatted_key = key.replace('_', ' ').replace('**', '').title()
                        items = ', '.join(str(v) for v in value)
                        paragraphs.append(f"{formatted_key}: {items}")
                    else:
                        formatted_key = key.replace('_', ' ').replace('**', '').title()
                        paragraphs.append(f"{formatted_key}: {value}")
                return '\n\n'.join(paragraphs)
    except (json.JSONDecodeError, ValueError):
        pass
    
    # Clean up markdown formatting
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)  # Remove **bold**
    text = re.sub(r'#{1,6}\s+', '', text)  # Remove markdown headers
    text = text.strip()
    
    return text

In [100]:
#Generic Agent for testing the ReAct agent with different queries and seeing the reasoning steps and final answer
def medibot_react(user_query: str, session_id: str) -> dict:
    """
    MediBot ReAct agent that reasons step-by-step and picks the right tools.
    
    Args:
        user_query (str): User's medical question or symptom description
    
    Returns:
        dict: Contains 'output' (final answer) and 'steps' (reasoning trace)
    """

    memory = get_memory(session_id)  # Fetch chat history for the session
    
    try:
        response = react_agent_with_chat_history.invoke(
            {"input": user_query, "chat_history": memory.messages},
            config={"configurable": {"session_id": session_id}}
        )
        
        steps_summary = []
        for action, observation in response.get("intermediate_steps", []):
            steps_summary.append({
                "tool": action.tool,
                "input": action.tool_input,
                "observation_preview": str(observation)[:200] + "..."
            })

        # Extract only the 'output' field from the response
        if isinstance(response, dict) and "output" in response:
            answer = response["output"]
        
            # Clean response to ensure natural English format
            answer = clean_response(answer)
            
            # Update memory with the conversation (LangChain handles this automatically, but we can also do it manually)
            memory.add_user_message(user_query)
            memory.add_ai_message(answer)
            
            return {"output": answer, "steps": steps_summary}
        else:
            return {"output": "Error: Unexpected response format", "steps": steps_summary}

    except Exception as e:
        return {"output": f"Sorry, I encountered an error: {str(e)}. Please try again.", "steps": []}


In [101]:
# Test the ReAct agent with different queries to see the reasoning steps and final answer
print("=" * 70)
print("MediBot ReAct Agent - Reasoning & Tool Selection Test")
print("=" * 70)

test_queries = [
    "I have high fever, joint pain, and rashes. What disease could this be and how serious is it?",
    "Explain what Malaria is and what precautions I should take",
    "I have chest pain and shortness of breath - is this urgent?"
]

for query in test_queries:
    print(f"\n{'=' * 70}")
    print(f"Query: {query}")
    print("=" * 70)
    result = medibot_react(query,session_id="test-session")
    print("\n--- Reasoning Steps ---")
    for i, step in enumerate(result["steps"], 1):
        print(f"Step {i}: Tool='{step['tool']}' | Input='{step['input']}' | Observation Preview='{step['observation_preview']}'")
    print("\n--- Final Answer ---")
    print(result)


MediBot ReAct Agent - Reasoning & Tool Selection Test

Query: I have high fever, joint pain, and rashes. What disease could this be and how serious is it?


> Entering new AgentExecutor chain...
The user is describing symptoms of high fever, joint pain, and rashes. I should first predict the probable disease based on these symptoms and then analyze the severity to guide the urgency of medical attention.
Action: predict_disease
Action Input: "high fever, joint pain, rashes"1. **Most Probable Disease(s)**: Dengue Fever
2. **Confidence**: High
3. **Key Matching Symptoms**: High fever, joint pain, skin rashThe predicted disease based on your symptoms of high fever, joint pain, and rashes is Dengue Fever with high confidence. Now, I should analyze the severity of these symptoms to guide the urgency of medical attention.
Action: analyze_symptom_severity
Action Input: "high fever, joint pain, rashes"1. **Severity Level per Symptom**:
   - High fever (severity: 7.0) - Severe
   - Joint pain (s

In [102]:
# Create Gradio app interface for the generic ReAct Medibot Agent
import gradio as gr

with gr.Blocks() as app:
    gr.Markdown("# 🤖 MediBot - Agents & ReAct Framework")
    gr.Markdown("Enter your symptoms below and get AI-powered responses with session memory.")

    with gr.Row():
        input_box = gr.Textbox(label="Enter your symptoms:", placeholder="Describe your symptoms...")
        output_box = gr.Textbox(label="Response:", lines=10)

    submit_button = gr.Button("Submit")

    # input_box = "Always provide your final answer as clear, conversational English text. Do not include JSON, code blocks, or tool metadata in your response."

    submit_button.click(medibot_react, inputs=input_box, outputs=output_box)

# Launch the Gradio app
app.launch(debug=True, share=True)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gradio/utils.py:1199: UserWarning: Expected 2 arguments for function <function medibot_react at 0x160005120>, received 1.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gradio/utils.py:1203: UserWarning: Expected at least 2 arguments for function <function medibot_react at 0x160005120>, received 1.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://9cb6cfa1884a4b4085.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gradio/helpers.py:1083: UserWarning: Unexpected argument. Filling with None.
  warnings.warn("Unexpected argument. Filling with None.")




> Entering new AgentExecutor chain...
The user is experiencing severe pain in the body and high fever, which could indicate a serious condition. I should predict the probable disease based on these symptoms and assess the urgency of the situation.
Action: predict_disease
Action Input: "severe pain in body and high fever"1. **Most Probable Disease(s)**: Dengue Fever
2. **Confidence**: High
3. **Key Matching Symptoms**: High fever, severe body painThe user is likely suffering from Dengue Fever based on the symptoms of severe pain in the body and high fever. Given the seriousness of this disease, immediate medical attention is necessary. I should provide information on the disease and suggest precautions to manage the symptoms.
Action: explain_disease
Action Input: "Dengue Fever"1. **What is it?**: Dengue Fever is a sickness caused by a virus spread by mosquitoes. It can make you feel really sick with headaches, severe joint pain, and a rash.

2. **Who gets it?**: People who live in or 

In [ ]:
import sys
import importlib
import json
import re

# Clear any cached gradio imports
for module in list(sys.modules.keys()):
    if 'gradio' in module:
        del sys.modules[module]

from langchain_classic.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import gradio as gr
from langchain_core.messages import HumanMessage, AIMessage

# Helper function to convert JSON to natural English
def clean_response(text):
    """
    Convert JSON format or structured responses to natural English text.
    """
    try:
        # Try to parse as JSON
        if text.strip().startswith('{') or text.strip().startswith('['):
            data = json.loads(text)
            if isinstance(data, dict):
                # Convert dict to readable paragraphs
                paragraphs = []
                for key, value in data.items():
                    if isinstance(value, list):
                        formatted_key = key.replace('_', ' ').replace('**', '').title()
                        items = ', '.join(str(v) for v in value)
                        paragraphs.append(f"{formatted_key}: {items}")
                    else:
                        formatted_key = key.replace('_', ' ').replace('**', '').title()
                        paragraphs.append(f"{formatted_key}: {value}")
                return '\n\n'.join(paragraphs)
    except (json.JSONDecodeError, ValueError):
        pass
    
    # Clean up markdown formatting
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)  # Remove **bold**
    text = re.sub(r'#{1,6}\s+', '', text)  # Remove markdown headers
    text = text.strip()
    
    return text

# Define the tools list
tools = [predict_disease, analyze_symptom_severity, explain_disease, suggest_precautions]

# Create the agent prompt
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are MediBot, a helpful medical assistant AI. You help users by:
    - Predicting probable diseases based on symptoms
    - Analyzing symptom severity and urgency
    - Explaining diseases in simple, understandable terms
    - Suggesting preventive measures and self-care recommendations

    Use the appropriate tool(s) based on the user's query:
    - Use 'predict_disease' when the user describes symptoms and wants to know what disease they might have
    - Use 'analyze_symptom_severity' when the user wants to know how serious their symptoms are
    - Use 'explain_disease' when the user wants to learn about a specific disease
    - Use 'suggest_precautions' when the user wants advice on prevention or self-care

    IMPORTANT: Always provide your final answer as natural, conversational English text.
    Do NOT use JSON, structured lists, code blocks, or technical formatting.
    Write like you're having a helpful, friendly conversation with the user.
    You may use multiple tools if needed to provide a comprehensive response.
    Always remind users to consult a healthcare professional for proper medical advice.
    """),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Use gpt-4 or gpt-3.5-turbo with tools
agent_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Create the agent
agent = create_openai_tools_agent(agent_llm, tools, agent_prompt)

# Create the agent executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

# Chat history for multi-turn conversations (stored as list of LangChain messages)
agent_chat_history = []

def medibot_chat(user_message):
    """
    MediBot chat function for Gradio interface.
    
    Args:
        user_message (str): User input message
    
    Returns:
        str: MediBot response in natural English
    """
    try:
        response = agent_executor.invoke({
            "input": user_message,
            "chat_history": agent_chat_history
        })
        answer = response["output"]
        
        # Clean response to ensure natural English format
        answer = clean_response(answer)
        
        # Update agent's internal chat history
        agent_chat_history.append(HumanMessage(content=user_message))
        agent_chat_history.append(AIMessage(content=answer))
        
        return answer
    except Exception as e:
        return f"Sorry, I encountered an error: {str(e)}. Please try again."


# Create Gradio interface
with gr.Blocks(title="MediBot - Medical Assistant", theme=gr.themes.Soft()) as medibot_ui:
    gr.Markdown("""
    # 🏥 MediBot - AI Medical Assistant
    ### Powered by RAG + LangChain Agent
    
    **I can help you with:**
    - 🔍 **Disease Prediction** - Describe your symptoms
    - ⚠️ **Severity Analysis** - Understand how serious your symptoms are  
    - 📚 **Disease Explanation** - Learn about a specific disease
    - 🛡️ **Precautions & Self-Care** - Get preventive recommendations
    
    > ⚕️ *Always consult a healthcare professional for proper medical diagnosis and treatment.*
    """)
    
    chatbot = gr.Chatbot(
        label="MediBot",
        height=500
    )
    
    with gr.Row():
        msg_input = gr.Textbox(
            label="Your message",
            placeholder="e.g., I have fever, headache and joint pain. What disease could this be?",
            scale=4
        )
        submit_btn = gr.Button("Send 💬", variant="primary", scale=1)
    
    with gr.Row():
        clear_btn = gr.Button("Clear Chat 🗑️", variant="secondary")
    
    gr.Examples(
        examples=[
            ["I have itching, skin rash and nodal skin eruptions. What disease might I have?"],
            ["How severe are symptoms like high fever, vomiting, and headache?"],
            ["Can you explain what Dengue is in simple terms?"],
            ["What precautions should I take for Fungal infection?"],
            ["I have chest pain, shortness of breath and sweating. Is this serious?"],
        ],
        inputs=msg_input,
        label="Example Questions"
    )
    
    def respond(message, chat_history_ui):
        # Get response from MediBot
        bot_response = medibot_chat(message)
        # Append to Gradio chat history as (user, bot) tuples
        chat_history_ui.append((message, bot_response))
        return "", chat_history_ui
    
    msg_input.submit(respond, [msg_input, chatbot], [msg_input, chatbot])
    submit_btn.click(respond, [msg_input, chatbot], [msg_input, chatbot])
    clear_btn.click(lambda: [], outputs=[chatbot], queue=False).then(
        lambda: agent_chat_history.clear(), queue=False
    )

try:
    medibot_ui.launch(share=False, debug=False)
except Exception as e:
    print(f"Error launching Gradio: {e}")
    print("Trying alternative launch method...")
    medibot_ui.launch(show_error=True, server_name="127.0.0.1", server_port=7860)


/var/folders/v2/pg84xz651xg2hdm_2qvdqd7h0000gp/T/ipykernel_50138/3372064702.py:123: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="MediBot - Medical Assistant", theme=gr.themes.Soft()) as medibot_ui:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.




> Entering new AgentExecutor chain...

Invoking: `predict_disease` with `{'symptoms': 'severe headache, high fever'}`


1. **Most Probable Disease(s)**: Meningitis
2. **Confidence**: High
3. **Key Matching Symptoms**: Severe headache, high fever
Invoking: `analyze_symptom_severity` with `{'symptoms': 'severe headache, high fever'}`


1. **Severity Level per Symptom**:
   - Severe headache (severity: 3.0) - Mild
   - High fever (severity: 7.0) - Severe

2. **Overall Urgency**: High urgency for seeking medical care

3. **Urgency Reasoning**: The combination of severe headache and high fever can indicate a potentially serious underlying condition that requires immediate medical attention. High fever can be a sign of infection or other serious health issues, and when combined with severe headache, it could indicate conditions such as meningitis or encephalitis.

4. **Immediate Action**: The user should seek medical attention immediately. It is important to consult a healthcare profession

Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gradio/queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gradio/route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gradio/blocks.py", line 2184, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gradio/blocks.py", line 1953, in postprocess_data
    prediction_value = await anyio.to_thread.run_sync(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Framewor



> Entering new AgentExecutor chain...

Invoking: `predict_disease` with `{'symptoms': 'itching, skin rash, nodal skin eruptions'}`


1. **Most Probable Disease(s)**: Dermatitis
2. **Confidence**: High
3. **Key Matching Symptoms**: itching, skin rash, nodal skin eruptionsBased on your symptoms of itching, skin rash, and nodal skin eruptions, the most probable disease is Dermatitis with a high level of confidence. Dermatitis is characterized by inflammation of the skin, leading to symptoms like itching, skin rash, and nodal skin eruptions.

Dermatitis is a common condition that can be caused by various factors such as allergies, irritants, or genetic predisposition. It is essential to avoid scratching the affected areas to prevent further irritation and potential infection.

For self-care, you can try the following:
1. Keep the affected area clean and dry.
2. Avoid known triggers or irritants that worsen the symptoms.
3. Use mild, fragrance-free moisturizers to soothe the skin.
4. Cons

Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gradio/queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gradio/route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gradio/blocks.py", line 2184, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gradio/blocks.py", line 1953, in postprocess_data
    prediction_value = await anyio.to_thread.run_sync(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Framewor